In [1]:
# import packages
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ["JAVA_HOME"] ="/home/manjari/Downloads/Fiji.app/java/linux-amd64/zulu8.60.0.21-ca-fx-jdk8.0.322-linux_x64/jre/lib/amd64/server/"
import json
import math
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import napari
# import IPython.display
import bardensr
import bardensr.plotting
# import imagej
import copy
from datetime import datetime
import shutil
import imagej
from cellpose import plot
from cellpose import utils, io
import tifffile
import argparse
import warnings
warnings.simplefilter('ignore', pd.errors.SettingWithCopyWarning)
warnings.simplefilter(action='ignore', category=FutureWarning)

import starmap.io as io
import starmap.preprocessing as preproc
import starmap.gene_calling as genecall
import starmap.cell_segmentation as cellseg
from starmap.config import Config


import IPython.display
import PIL
from PIL import Image, ImageDraw

In [2]:
# read in config ---------------------------------------------------------------
config = Config(filepath_homedir = "/home/manjari/Documents/ruitao/pipeline_run/MA27/",
                filepath_rawdata = "/run/user/1006/gvfs/afp-volume:host=KebschullLab1.local,user=Dylan,volume=common_butterwort/manjari/MA27/1000gene_set_BSPEG_02_25_2024/Manjari_e13_coronal_02_25_2024_1000genes_BSPEG/",
                filepath_codebook = "/run/user/1006/gvfs/afp-volume:host=KebschullLab1.local,user=Dylan,volume=common_butterwort/manjari/MA27/analysis/Mouse_TFs_Prim_barcode_10_17_22 (2).csv",
                filepath_dapi_all_z = "/run/user/1006/gvfs/afp-volume:host=KebschullLab1.local,user=Dylan,volume=common_butterwort/manjari/MA27/1000gene_set_BSPEG_02_25_2024/Manjari_e13_coronal_02_25_2024_1000genes_BSPEG/round1/dapi_all_z.nd2",
                filepath_dapi = "/run/user/1006/gvfs/afp-volume:host=KebschullLab1.local,user=Dylan,volume=common_butterwort/manjari/MA27/1000gene_set_BSPEG_02_25_2024/Manjari_e13_coronal_02_25_2024_1000genes_BSPEG/round1/dapi.nd2",
                filepath_nissl = "",
                round_index = [1, 2, 3, 4, 5, 6, 7],
                fov_num = 51,
                fov_align = 26,
                fov_minmax = [33, 26, 21, 14, 9, 8, 15, 20, 27, 32, 31, 28, 19, 16, 7],
                # cellpose_model = 'TN3',
                # cellpose_channel = [2, 3],
                ifov = 26,
                igenes = ["Slc17a6", "Gad1", "Atoh1", "Ptf1a", "Meis2", "Pax5", "Pax2", "Tbr1", "Eomes", "Sox14", "nogene1", "nogene2", "nogene3", "nogene4", "nogene5"],
                SpGene = "Finished",
                SpMask = "RmvOverlap",
                grid_stitch = True,
                # bardensr_reg = True,
                find_param = True,
                remove_overlap = True
                )


In [ ]:
# load in data ---------------------------------------------------------------
print(datetime.now().strftime("%d/%m/%Y %H:%M:%S"), "   load in data")
# dapi_images = np.load(filepath_loadimg+'dapi.npy')
# [nissl_images, _] = io.read_dapi_images(filepath_nissl)
# merged_images = []
# for i in range(len(dapi_images)):
#     merged_images.append(np.stack([np.zeros(dapi_images[i].shape), nissl_images[i]/np.max(nissl_images[i]), dapi_images[i]/np.max(dapi_images[i])], axis=2))

# codebook, genenames, codeflat = genecall.create_codebook(config.filepath_codebook, config.round_index0, config.base_code)
# position_reg = pd.read_csv(filepath_loadimg + 'position_reg.csv')
image_preped = io.open_hdf5(config.filepath_output+'image_preped.hdf5')
# image_registered = io.open_hdf5(config.filepath_output+'image_registered.hdf5')
# image_aligned = io.open_hdf5(config.filepath_output+'image_aligned.hdf5')
# gene_called = pd.read_csv(config.filepath_output+'gene_called.csv')
# gene_mapped = pd.read_csv(filepath_output+'gene_mapped.csv')
# gene_trimmed = pd.read_csv(filepath_output+'gene_trimmed.csv')
# mask_mapped = io.open_hdf5(filepath_output+'mask_mapped.hdf5')
# mask_trimmed = io.open_hdf5(filepath_output+'mask_trimmed.hdf5')

# image_corrected = io.open_hdf5(filepath_output+'image_corrected.hdf5')
# image_normed = io.open_hdf5(filepath_output+'image_normed.hdf5')


### check if dots stay bright across rounds

In [5]:
image_registered = io.open_hdf5_NxRxC(config.filepath_output+'image_registered_swaped.hdf5', config.fov_num, config.round_num)
gene_called = pd.read_csv(config.filepath_output+'gene_called.csv')

start loading: /home/manjari/Documents/ruitao/pipeline_run/MA27/STARmap_output/image_registered_swaped.hdf5
1.9607843137254901% done loading hdf5 file
3.9215686274509802% done loading hdf5 file
5.882352941176471% done loading hdf5 file
7.8431372549019605% done loading hdf5 file
9.803921568627452% done loading hdf5 file
11.764705882352942% done loading hdf5 file
13.72549019607843% done loading hdf5 file
15.686274509803921% done loading hdf5 file
17.647058823529413% done loading hdf5 file
19.607843137254903% done loading hdf5 file
21.568627450980394% done loading hdf5 file
23.529411764705884% done loading hdf5 file
25.49019607843137% done loading hdf5 file
27.45098039215686% done loading hdf5 file
29.41176470588235% done loading hdf5 file
31.372549019607842% done loading hdf5 file
33.333333333333336% done loading hdf5 file
35.294117647058826% done loading hdf5 file
37.254901960784316% done loading hdf5 file
39.21568627450981% done loading hdf5 file
41.1764705882353% done loading hdf5 fil

In [6]:
# obtain intensity
intensity_avg = []
for FOV in range(config.fov_num):
    image_fov = np.max(image_registered[FOV], axis=1)
    gene_fov = gene_called[gene_called['FOV']==FOV]
    intensity = image_fov[:, gene_fov['m1'], gene_fov['m2']]
    intensity_avg.append(np.average(intensity, axis=1))
intensity_avg = np.array(intensity_avg)

In [8]:
config.fov_num

51

In [11]:
# plot
col = 5
row = math.ceil(config.fov_num / col)
plt.figure(figsize=(5*col, 5*row))
for FOV in range(config.fov_num):
    plt.subplot(row, col, FOV+1)
    plt.plot(config.round_index, intensity_avg[FOV])
    plt.title('FOV %s' % str(FOV))
    plt.ylim(np.min(intensity_avg)-0.01, np.max(intensity_avg)+0.01)
    plt.xlabel('round')
    plt.ylabel('pixel value')

### Check if spots stay present across rounds

In [ ]:
image_registered = io.open_hdf5_NxRxC(config.filepath_output+'image_registered.hdf5', config.fov_num, config.round_num)

In [ ]:
# blob detection with skimage
from math import sqrt
from skimage import data
from skimage.feature import blob_log, blob_dog, blob_doh

FOV = 1
R = 0
image_fov = image_registered[FOV]
image_round = np.max(image_fov[R], axis=0)
blobs_log = blob_log(image_round, max_sigma=30, num_sigma=10, threshold=.1)
blobs_dog = blob_dog(image_round, max_sigma=30, threshold=.1)
blobs_doh = blob_doh(image_round, max_sigma=30, threshold=.01)

In [ ]:
plt.figure(figsize=(20, 20))
plt.subplot(2, 2, 1)
plt.imshow(image_round)
plt.title('original')
plt.subplot(2, 2, 2)
plt.imshow(image_round)
plt.scatter(blobs_log[:, 1], blobs_log[:, 0], s=0.1, marker='o', color='none', edgecolors='red')
plt.title('log')
plt.subplot(2, 2, 3)
plt.imshow(image_round)
plt.scatter(blobs_dog[:, 1], blobs_dog[:, 0], s=0.1, marker='o', color='none', edgecolors='red')
plt.title('dog')
plt.subplot(2, 2, 4)
plt.imshow(image_round)
plt.scatter(blobs_doh[:, 1], blobs_doh[:, 0], s=0.1, marker='o', color='none', edgecolors='red')
plt.title('doh')
plt.tight_layout()
plt.show()

In [ ]:
# blob detection with cv2
import cv2
import numpy as np
from PIL import Image

# Read image
# im = cv2.imread("blob.jpg", cv2.IMREAD_GRAYSCALE)
FOV = 1
R = 0
image_fov = image_registered[FOV]
image_round = np.uint8(np.max(image_fov[R], axis=0)*256)
print(image_round.shape)
im = Image.fromarray(image_round)
# Setup SimpleBlobDetector parameters
params = cv2.SimpleBlobDetector_Params()
# Change thresholds
params.minThreshold = 1
params.maxThreshold = 200
# Filter by Area.
params.filterByArea = True
params.minArea = 0.1
# Filter by Circularity
params.filterByCircularity = True
params.minCircularity = 0.1
# Filter by Convexity
params.filterByConvexity = True
params.minConvexity = 0.87
# Filter by Inertia
params.filterByInertia = True
params.minInertiaRatio = 0.01
# Create a detector with the parameters
detector = cv2.SimpleBlobDetector_create(params)
# # Set up the detector with default parameters.
# detector = cv2.SimpleBlobDetector_create()
# Detect blobs.
keypoints = detector.detect(image_round)
 
# Draw detected blobs as red circles.
blank = np.zeros((1, 1))
blobs = cv2.drawKeypoints(image_round, keypoints, blank, (0, 0, 255), cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
# cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS ensures the size of the circle corresponds to the size of blob
# im_with_keypoints = cv2.drawKeypoints(image_round, keypoints, np.array([]), (0,0,255), cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
plt.figure(figsize=(30,30))
plt.imshow(blobs)
 
# # Show keypoints
# cv2.imshow("Keypoints", im_with_keypoints)
# cv2.waitKey(0)

### Check how many incorrect genes are there?

In [ ]:
gene_called = pd.read_csv(config.filepath_output+'gene_called.csv')
codebook, genenames, codeflat = genecall.create_codebook(config.filepath_codebook, config.round_index0, config.base_code)

In [ ]:
nogene_list = [s for s in genenames if "nogene" in s]
percentage = []
for NOGENE in nogene_list:
    percentage_nogene = []
    for FOV in range(config.fov_num):
        gene_fov = gene_called[gene_called['FOV']==FOV]
        gene_nogene = gene_fov[gene_fov['Names']==NOGENE]
        percentage_nogene.append((len(gene_nogene))/(len(gene_fov)+0.01)*100)
    percentage.append(percentage_nogene)

percentage.append(np.sum(percentage, axis=0))
nogene_list.append('all nogenes')

In [ ]:
# plot
col = 3
row = int(len(nogene_list) / col)
plt.figure(figsize=(col*10, row*5))
for i in range(len(nogene_list)):
    plt.subplot(row, col, i+1)
    plt.bar(np.arange(config.fov_num), percentage[i])
    plt.title(nogene_list[i])
    plt.xlabel('fov')
    plt.ylabel('percentage of no (%)')

### Check fdr for each fov

In [ ]:
gene_called = pd.read_csv(config.filepath_output+'gene_called.csv')
codebook, genenames, codeflat = genecall.create_codebook(config.filepath_codebook, config.round_index0, config.base_code)

In [ ]:
def qc_genecalling(gene_result, genenames):
    nogene_names = [s for s in genenames if "nogene" in s]
    trgene_names = [s for s in genenames if "nogene" not in s]
    nogene_result = gene_result.iloc[np.where(np.isin(gene_result['Names'],nogene_names))]
    trgene_result = gene_result.iloc[np.where(np.isin(gene_result['Names'],trgene_names))]
    fdr_overall = (len(nogene_result)+0.01)/(len(np.unique(nogene_result['Names']))+0.01)*(len(np.unique(trgene_result['Names']))+0.01)/(len(trgene_result)+0.01)
    print(len(nogene_result), len(trgene_result))
    fdr_list, fov_list = [], []
    for i in range(int(np.unique(gene_result['FOV'])[-1])+1):
        nogene_result_fov = nogene_result[nogene_result['FOV']==i]
        trgene_result_fov = trgene_result[trgene_result['FOV']==i]
        fdr = (len(nogene_result_fov)+0.01)/(len(np.unique(nogene_result_fov['Names']))+0.01)*(len(np.unique(trgene_result_fov['Names']))+0.01)/(len(trgene_result_fov)+0.01)
        # if fdr < 0.8:
        fdr_list.append(fdr)
        fov_list.append(i)
    print(fdr_list)
    print(fov_list)
    # fdr_mean = np.mean(fdr_list)
    print(fdr_overall)
    return fov_list, fdr_list, fdr_overall
fov_list, fdr_list, fdr_overall = qc_genecalling(gene_called, genenames)

In [ ]:
plt.scatter(fov_list, fdr_list, color='lightblue')
plt.ylim(0, np.max(fdr_list)+0.1)
plt.axhline(fdr_overall, color='lightcoral')
plt.title("FDR Across FOV")
plt.xlabel('fov')
plt.ylabel('fdr')

### Check how many spots per cell and how many unique genes per cell

In [ ]:
gene_mapped = pd.read_csv(config.filepath_output+'gene_mapped.csv')

In [ ]:
gene2cell_fov, unigene2cell_fov = [], []
for FOV in range(config.fov_num):
    gene_fov = gene_mapped[gene_mapped['FOV']==FOV]
    gene_cell, unigene_cell = [], []
    for CELL in np.unique(gene_fov['cell_number']):
        temp = gene_fov[gene_fov['cell_number']==CELL]
        gene_cell.append(len(temp))
        unigene_cell.append(len(np.unique(temp['Names'])))
    gene2cell_fov.append(np.average(gene_cell))
    unigene2cell_fov.append(np.average(unigene_cell))

In [ ]:
plt.figure(figsize=(int(config.fov_num*0.2)*2, int(config.fov_num*0.2)))
bar1 = plt.bar(x=np.arange(config.fov_num), height=gene2cell_fov, width=0.4, label='genes per cell', color='lightblue', tick_label=range(config.fov_num))
bar2 = plt.bar(x=np.arange(config.fov_num)+0.4, height=unigene2cell_fov, width=0.4, label='unique gene per cell', color='lightcoral')
# plt.bar_label(bar1)
# plt.bar_label(bar2)
plt.title('genes per cell & unique gene per cell')
plt.xlabel('fov')
plt.ylabel('num')
plt.xticks(np.arange(config.fov_num)+0.2, range(config.fov_num))
plt.legend()


### check aligned/registered images for specific fov

In [ ]:
# check aligned images
FOV = config.ifov
IPython.display.Image(bardensr.plotting.gify(image_aligned[FOV][:,:,500:1000, 500:1000],normstyle='each', duration=500),width=500)

In [ ]:
# use napari to visualize aligned/registered images
FOV = ifov
imgs = np.array([bardensr.plotting.lutup(*x,sc=.5,normstyle='each') for x in image_aligned[FOV][:,:,500:1000,500:1000]])
viewer = napari.view_image(imgs, rgb=True)

### check aligned/registered images for specific fov with gene calls

In [ ]:
# check aligned images with gene calls
FOV = ifov
IMG = io.display_spot2gene(gene_called[gene_called['FOV']==FOV], genenames, image_aligned[FOV])
IPython.display.Image(bardensr.plotting.gify(IMG[:,:,500:600,500:600], normstyle='each', duration=1000),width=500)


In [ ]:
# use napari to visualize aligned images (* with gene calls)
FOV = ifov
IMG = io.display_spot2gene(gene_called[gene_called['FOV']==FOV], genenames, image_aligned[FOV])
imgs = np.array([bardensr.plotting.lutup(*x,sc=.5,normstyle='each') for x in IMG[:,:,500:1000,500:1000]])
viewer = napari.view_image(imgs, rgb=True)

In [ ]:
# outlines = utils.masks_to_outlines(mask_mapped[ifov])
# overlay = plot.mask_overlay(dapi_images[ifov], mask_mapped[ifov])
# mask_color = plot.mask_rgb(mask_mapped[ifov])

In [ ]:
# plot gene spots before & after mapping to cells ----------------------
print(datetime.now().strftime("%d/%m/%Y %H:%M:%S"), "----plot gene spots before & after mapping to cells")
plt.figure(figsize=(30,10))
plt.subplot(1,3,1)
plt.imshow(merged_images[ifov])
gene_ifov = gene_called[gene_called['FOV']==ifov]
for i in range(len(genenames)):
    gene_pos = gene_ifov[gene_ifov['Names'] == genenames[i]]
    plt.scatter(gene_pos['m2'], gene_pos['m1'], s=0.3, label=genenames[i], cmap='viridis')
plt.title('Before map gene to cell')
plt.subplot(1,3,2)
plt.imshow(merged_images[ifov])
gene_ifov = gene_mapped[gene_mapped['FOV']==ifov]
for i in range(len(genenames)):
    gene_pos = gene_ifov[gene_ifov['Names'] == genenames[i]]
    plt.scatter(gene_pos['m2'], gene_pos['m1'], s=0.3, label=genenames[i], cmap='viridis')
plt.title('After map gene to cell')
plt.subplot(1,3,3)
plt.imshow(merged_images[ifov])
plt.imshow(mask_color, alpha=0.5)
plt.tight_layout()
plt.title('Cell segmentation masks')

In [ ]:
# save registration.gif
imgs=[bardensr.plotting.lutup(*x,sc=.5,normstyle='each') for x in image_aligned[FOV][:,:,500:1000, 500:1000]]
# imgs=[PIL.Image.fromarray(x) for x in imgs]
# imgs[0].save(filepath_val+'registration.gif', save_all=True, append_images=imgs[1:], optimize=False, duration=500, loop=0)
tifffile.imsave(filepath_val+'registration.tif', np.array(imgs))

In [ ]:
# histogram plot of pixel values, value range (0, 1)
his_1, bin_1 = np.histogram(image_aligned[ifov][0, 0], bins=1000, range=(0, 1))
# his_2, bin_2 = np.histogram(image_aligned[ifov][0, 0], bins=256, range=(0, 1))
plt.figure()
plt.title("Intensity Histogram")
plt.xlabel("pixel value")
plt.ylabel("pixel count")
plt.plot(bin_1[0:-1], his_1, color='lightblue', label = "image_1") # color='lightcoral', 'lightgreen'
# plt.xlim([-0.1, 1.0])
# plt.ylim([-10000, 700000])
# plt.xlim([0.95, 1.0])
# plt.ylim([0.0, 2500])
plt.legend()

In [ ]:
# plot cell masks with cell ID
FOV = ifov
plt.figure(figsize=(10, 10))
overlay2 = plot.mask_overlay(dapi_images[FOV], mask_trimmed[FOV])
plt.imshow(overlay2, alpha=0.8)
cell_trimmed = gene_trimmed.drop_duplicates(subset=['cell_number'], keep='first', inplace=False)
cell_temp = cell_trimmed[cell_trimmed['FOV']==FOV]
plt.scatter(cell_temp['cell_y'], cell_temp['cell_x'], c='black', s=0.3, cmap='viridis')
for index, row in cell_temp.iterrows():
    plt.annotate(row['cell_number'], xy=(row['cell_y'], row['cell_x']), fontsize=5)

In [ ]:
# plot global cell masks, diff color in fov
plt.figure(figsize=(30,30))
mask_onenum = copy.deepcopy(mask_trimmed)
for i in range(len(mask_onenum)):
    mask_onenum[i][np.where(mask_onenum[i]!=0)] = i
full_cell_mask = io.load_mask(mask_onenum, position_reg, 2304)
overlay = plot.mask_rgb(full_cell_mask)
plt.imshow(overlay, alpha=0.8)
plt.scatter(cell_trimmed['center_y'], cell_trimmed['center_x'], marker=',', c='grey', s=0.1, cmap='viridis')

In [ ]:
# # not used
# # plot cell center over dapi images -----------------------------
# print(datetime.now().strftime("%d/%m/%Y %H:%M:%S"), "----plot cell center over dapi images")
# full_dapi_image = io.load_dapi(dapi_images, position_reg, 2304)
# plt.figure(figsize=(30,30))
# plt.imshow(full_dapi_image, vmin = 0, vmax = 200,alpha=0.3)
# for i in np.unique(gene_mapped['FOV']):
#     gene_temp = gene_mapped[gene_mapped['FOV']==i]
#     plt.scatter(gene_temp['center_y'], gene_temp['center_x'], s=0.3, label=i, cmap='viridis')
# # plt.scatter(gene_mapped['center_y'], gene_mapped['center_x'], s=0.3, c = 'red', cmap='viridis')
# plt.savefig(filepath_val+'cell_center_over_dapi.png')

# # plot cell center over cell masks -----------------------------
# print(datetime.now().strftime("%d/%m/%Y %H:%M:%S"), "----plot cell center over cell masks")
# full_cell_mask = io.load_dapi(mask_trimmed, position_reg, 2304)
# plt.figure(figsize=(30,30))
# plt.imshow(full_cell_mask)
# for i in np.unique(gene_mapped['FOV']):
#     gene_temp = gene_mapped[gene_mapped['FOV']==i]
#     plt.scatter(gene_temp['center_y'], gene_temp['center_x'], s=0.3, label=i, cmap='viridis')
# # plt.scatter(gene_mapped['center_y'], gene_mapped['center_x'], s=0.3, c = 'red', cmap='viridis')
# plt.savefig(filepath_val+'cell_center_over_mask.png')